In [3]:
import sys
sys.path.append("../../")
import pandas as pd

df_stroke = pd.read_csv(r"../data/wids_pre/wids.csv")

In [4]:
from callmefair.search._search_base import CType, combine_attributes

combined_df = combine_attributes(df_stroke, cols=['race', 'payer'], operation=CType.intersection)
combined_df

c:\Users\enemy\anaconda3\Lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
c:\Users\enemy\anaconda3\Lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg = vmap(vect_normalized_disc

,patient_age,breast_cancer_diagnosis_code,cancer_area,metastatic_cancer_diagnosis_code,population,female,married,income_household_150_over,income_household_six_figure,income_individual_median,rent_median,education_bachelors,education_graduate,DiagPeriodL90D,neighborhood_affluence,neighborhood_education,neighborhood_employment,neighborhood_majority_white,state_privileged,race_payer
0,0,45,21,2,31437.75000,50.142857,36.571429,7.528571,19.100000,24563.57143,1165.000000,8.357143,3.257143,1,0,0,0,0,1,0
1,0,27,26,0,39121.87879,50.106061,50.245455,29.596970,49.357576,41287.27273,2003.125000,23.739394,12.245455,1,1,1,1,0,1,1
2,0,16,3,0,21996.68333,49.876667,55.753333,18.680000,39.555000,40399.03333,1235.907407,19.678333,10.115000,1,1,0,1,1,0,1
3,0,20,22,0,32795.32558,50.933333,52.604762,38.057143,56.907143,55336.28571,2354.738095,33.285714,22.459524,0,1,1,1,0,1,1
4,0,7,21,0,10886.26000,47.688000,57.882000,8.606000,22.226000,29073.18367,919.743590,13.978000,5.684000,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12818,0,27,26,0,19413.05882,51.735294,36.429412,7.358824,18.352941,27888.52941,772.647059,14.400000,8.370588,1,0,0,0,0,0,1
12819,0,44,19,0,30153.87952,50.272840,53.076543,37.245000,55.248750,52778.65000,2223.445946,26.903704,18.277778,1,1,1,1,0,1,1
12820,0,44,19,2,32795.32558,50.933333,52.604762,38.057143,56.907143,55336.28571,2354.738095,33.285714,22.459524,1,1,1,1,0,1,1
12821,0,7,21,0,71374.13158,52.331579,39.923684,21.318421,36.207895,39491.78947,1678.447368,24.371053,16.655263,0,0,1,0,0,0,1


In [19]:
combined_df.to_csv('stroke_age_ever_married_Residence_type.csv',index=False)

In [3]:
from sklearn.model_selection import train_test_split
from callmefair.util.fair_util import BMInterface
from callmefair.mitigation.fair_bm import BMManager

df_train, df_test= train_test_split(combined_df, test_size=0.3, stratify=combined_df[['age_ever_married', 'stroke']] ,random_state=42)
df_test, df_val= train_test_split(df_test, test_size=0.5, stratify= df_test[['age_ever_married', 'stroke']],random_state=42)

# Defining the name of the label column
label_name = 'stroke'
# Define the name of the privileged group
sensitive_attribute = ['age_ever_married']

bm_interface = BMInterface(df_train, df_val, df_test, label_name, sensitive_attribute)

privileged_groups = [{'age_ever_married': 1}]
unprivileged_groups = [{'age_ever_married': 0}]

bm_manager = BMManager(bm_interface, privileged_groups, unprivileged_groups)

In [7]:
from pytorch_tabnet.tab_model import TabNetClassifier
from callmefair.mitigation.fair_grid import BMGridSearch
from callmefair.mitigation.fair_bm import BMType

tabnet = TabNetClassifier(seed=42)

bm_combinations = [
    [BMType.preReweighing],  # Only preprocessing
    [BMType.preDisparate],   # Only disparate impact remover
    [BMType.preReweighing, BMType.posCalibrated],  # Preprocessing + postprocessing
]

# Initialize grid search
grid_search = BMGridSearch(
    bmI=bm_interface,
    model=tabnet,
    bm_list=bm_combinations,
    privileged_group=privileged_groups,
    unprivileged_group=unprivileged_groups
)

# Run comprehensive evaluation
grid_search.run_single_sensitive()

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.4834  | val_0_auc: 0.53383 |  0:00:00s
epoch 1  | loss: 0.41312 | val_0_auc: 0.55016 |  0:00:01s
epoch 2  | loss: 0.40317 | val_0_auc: 0.58496 |  0:00:01s
epoch 3  | loss: 0.35987 | val_0_auc: 0.63319 |  0:00:02s
epoch 4  | loss: 0.35728 | val_0_auc: 0.6464  |  0:00:02s
epoch 5  | loss: 0.36207 | val_0_auc: 0.65585 |  0:00:03s
epoch 6  | loss: 0.37038 | val_0_auc: 0.65016 |  0:00:03s
epoch 7  | loss: 0.35324 | val_0_auc: 0.64458 |  0:00:04s
epoch 8  | loss: 0.36109 | val_0_auc: 0.64812 |  0:00:04s
epoch 9  | loss: 0.34772 | val_0_auc: 0.64952 |  0:00:05s
epoch 10 | loss: 0.33489 | val_0_auc: 0.65145 |  0:00:06s
epoch 11 | loss: 0.31783 | val_0_auc: 0.65897 |  0:00:06s
epoch 12 | loss: 0.3395  | val_0_auc: 0.67035 |  0:00:07s
epoch 13 | loss: 0.35211 | val_0_auc: 0.67277 |  0:00:07s
epoch 14 | loss: 0.3189  | val_0_auc: 0.66778 |  0:00:08s
epoch 15 | loss: 0.33549 | val_0_auc: 0.66101 |  0:00:08s
epoch 16 | loss: 0.30852 | val_0_auc: 0.6724  |  0:00:09s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.4834  | val_0_auc: 0.53383 |  0:00:00s
epoch 1  | loss: 0.41312 | val_0_auc: 0.55016 |  0:00:00s
epoch 2  | loss: 0.40317 | val_0_auc: 0.58496 |  0:00:01s
epoch 3  | loss: 0.35987 | val_0_auc: 0.63319 |  0:00:02s
epoch 4  | loss: 0.35728 | val_0_auc: 0.6464  |  0:00:03s
epoch 5  | loss: 0.36207 | val_0_auc: 0.65585 |  0:00:03s
epoch 6  | loss: 0.37038 | val_0_auc: 0.65016 |  0:00:04s
epoch 7  | loss: 0.35324 | val_0_auc: 0.64458 |  0:00:04s
epoch 8  | loss: 0.36109 | val_0_auc: 0.64812 |  0:00:05s
epoch 9  | loss: 0.34772 | val_0_auc: 0.64952 |  0:00:05s
epoch 10 | loss: 0.33489 | val_0_auc: 0.65145 |  0:00:06s
epoch 11 | loss: 0.31783 | val_0_auc: 0.65897 |  0:00:06s
epoch 12 | loss: 0.3395  | val_0_auc: 0.67035 |  0:00:07s
epoch 13 | loss: 0.35211 | val_0_auc: 0.67277 |  0:00:08s
epoch 14 | loss: 0.3189  | val_0_auc: 0.66778 |  0:00:08s
epoch 15 | loss: 0.33549 | val_0_auc: 0.66101 |  0:00:09s
epoch 16 | loss: 0.30852 | val_0_auc: 0.6724  |  0:00:09s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.48623 | val_0_auc: 0.52578 |  0:00:00s
epoch 1  | loss: 0.43286 | val_0_auc: 0.56928 |  0:00:01s
epoch 2  | loss: 0.41046 | val_0_auc: 0.60773 |  0:00:01s
epoch 3  | loss: 0.36619 | val_0_auc: 0.64887 |  0:00:02s
epoch 4  | loss: 0.35728 | val_0_auc: 0.67938 |  0:00:02s
epoch 5  | loss: 0.36079 | val_0_auc: 0.70924 |  0:00:03s
epoch 6  | loss: 0.36404 | val_0_auc: 0.70569 |  0:00:03s
epoch 7  | loss: 0.35284 | val_0_auc: 0.70892 |  0:00:04s
epoch 8  | loss: 0.34776 | val_0_auc: 0.7051  |  0:00:04s
epoch 9  | loss: 0.34263 | val_0_auc: 0.68942 |  0:00:05s
epoch 10 | loss: 0.32058 | val_0_auc: 0.68738 |  0:00:05s
epoch 11 | loss: 0.32765 | val_0_auc: 0.67857 |  0:00:06s
epoch 12 | loss: 0.32709 | val_0_auc: 0.68571 |  0:00:06s
epoch 13 | loss: 0.34798 | val_0_auc: 0.6899  |  0:00:07s
epoch 14 | loss: 0.31567 | val_0_auc: 0.70344 |  0:00:08s
epoch 15 | loss: 0.326   | val_0_auc: 0.71665 |  0:00:08s
epoch 16 | loss: 0.31926 | val_0_auc: 0.71751 |  0:00:09s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.4834  | val_0_auc: 0.53383 |  0:00:00s
epoch 1  | loss: 0.41312 | val_0_auc: 0.55016 |  0:00:01s
epoch 2  | loss: 0.40317 | val_0_auc: 0.58496 |  0:00:01s
epoch 3  | loss: 0.35987 | val_0_auc: 0.63319 |  0:00:02s
epoch 4  | loss: 0.35728 | val_0_auc: 0.6464  |  0:00:02s
epoch 5  | loss: 0.36207 | val_0_auc: 0.65585 |  0:00:03s
epoch 6  | loss: 0.37038 | val_0_auc: 0.65016 |  0:00:03s
epoch 7  | loss: 0.35324 | val_0_auc: 0.64458 |  0:00:04s
epoch 8  | loss: 0.36109 | val_0_auc: 0.64812 |  0:00:04s
epoch 9  | loss: 0.34772 | val_0_auc: 0.64952 |  0:00:05s
epoch 10 | loss: 0.33489 | val_0_auc: 0.65145 |  0:00:05s
epoch 11 | loss: 0.31783 | val_0_auc: 0.65897 |  0:00:06s
epoch 12 | loss: 0.3395  | val_0_auc: 0.67035 |  0:00:06s
epoch 13 | loss: 0.35211 | val_0_auc: 0.67277 |  0:00:07s
epoch 14 | loss: 0.3189  | val_0_auc: 0.66778 |  0:00:08s
epoch 15 | loss: 0.33549 | val_0_auc: 0.66101 |  0:00:08s
epoch 16 | loss: 0.30852 | val_0_auc: 0.6724  |  0:00:09s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


---

In [12]:
df_train, df_test= train_test_split(df_stroke, test_size=0.3, stratify=df_stroke[['age', 'stroke']] ,random_state=42)
df_test, df_val= train_test_split(df_test, test_size=0.5, stratify= df_test[['age', 'stroke']],random_state=42)

# Define the name of the privileged group
sensitive_attribute = ['age']
privileged_groups = [{'age': 1}]
unprivileged_groups = [{'age': 0}]

bm_interface = BMInterface(df_train, df_val, df_test, label_name, sensitive_attribute)
bm_manager = BMManager(bm_interface, privileged_groups, unprivileged_groups)

# Initialize grid search
grid_search = BMGridSearch(
    bmI=bm_interface,
    model=tabnet,
    bm_list=bm_combinations,
    privileged_group=privileged_groups,
    unprivileged_group=unprivileged_groups
)

# Run comprehensive evaluation
grid_search.run_single_sensitive()

epoch 0  | loss: 1.05944 | val_0_auc: 0.46946 |  0:00:00s
epoch 1  | loss: 0.73135 | val_0_auc: 0.44177 |  0:00:01s
epoch 2  | loss: 0.58987 | val_0_auc: 0.47231 |  0:00:01s
epoch 3  | loss: 0.4942  | val_0_auc: 0.51802 |  0:00:02s
epoch 4  | loss: 0.47412 | val_0_auc: 0.54548 |  0:00:02s
epoch 5  | loss: 0.43199 | val_0_auc: 0.57548 |  0:00:03s
epoch 6  | loss: 0.40901 | val_0_auc: 0.58668 |  0:00:03s
epoch 7  | loss: 0.39866 | val_0_auc: 0.64359 |  0:00:04s
epoch 8  | loss: 0.39068 | val_0_auc: 0.69172 |  0:00:05s
epoch 9  | loss: 0.38024 | val_0_auc: 0.72314 |  0:00:05s
epoch 10 | loss: 0.38659 | val_0_auc: 0.73665 |  0:00:06s
epoch 11 | loss: 0.35625 | val_0_auc: 0.73665 |  0:00:06s
epoch 12 | loss: 0.36189 | val_0_auc: 0.73478 |  0:00:07s
epoch 13 | loss: 0.35469 | val_0_auc: 0.7455  |  0:00:07s
epoch 14 | loss: 0.34389 | val_0_auc: 0.73396 |  0:00:08s
epoch 15 | loss: 0.36393 | val_0_auc: 0.73709 |  0:00:09s
epoch 16 | loss: 0.34382 | val_0_auc: 0.73698 |  0:00:09s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 1.05944 | val_0_auc: 0.46946 |  0:00:00s
epoch 1  | loss: 0.73135 | val_0_auc: 0.44177 |  0:00:01s
epoch 2  | loss: 0.58987 | val_0_auc: 0.47231 |  0:00:01s
epoch 3  | loss: 0.4942  | val_0_auc: 0.51802 |  0:00:02s
epoch 4  | loss: 0.47412 | val_0_auc: 0.54548 |  0:00:02s
epoch 5  | loss: 0.43199 | val_0_auc: 0.57548 |  0:00:03s
epoch 6  | loss: 0.40901 | val_0_auc: 0.58668 |  0:00:03s
epoch 7  | loss: 0.39866 | val_0_auc: 0.64359 |  0:00:04s
epoch 8  | loss: 0.39068 | val_0_auc: 0.69172 |  0:00:04s
epoch 9  | loss: 0.38024 | val_0_auc: 0.72314 |  0:00:05s
epoch 10 | loss: 0.38659 | val_0_auc: 0.73665 |  0:00:05s
epoch 11 | loss: 0.35625 | val_0_auc: 0.73665 |  0:00:06s
epoch 12 | loss: 0.36189 | val_0_auc: 0.73478 |  0:00:06s
epoch 13 | loss: 0.35469 | val_0_auc: 0.7455  |  0:00:07s
epoch 14 | loss: 0.34389 | val_0_auc: 0.73396 |  0:00:07s
epoch 15 | loss: 0.36393 | val_0_auc: 0.73709 |  0:00:08s
epoch 16 | loss: 0.34382 | val_0_auc: 0.73698 |  0:00:08s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 1.05856 | val_0_auc: 0.51093 |  0:00:00s
epoch 1  | loss: 0.7207  | val_0_auc: 0.4031  |  0:00:01s
epoch 2  | loss: 0.55438 | val_0_auc: 0.4037  |  0:00:01s
epoch 3  | loss: 0.47468 | val_0_auc: 0.50461 |  0:00:02s
epoch 4  | loss: 0.44899 | val_0_auc: 0.59904 |  0:00:02s
epoch 5  | loss: 0.40383 | val_0_auc: 0.63047 |  0:00:03s
epoch 6  | loss: 0.37889 | val_0_auc: 0.70045 |  0:00:03s
epoch 7  | loss: 0.3637  | val_0_auc: 0.74121 |  0:00:04s
epoch 8  | loss: 0.35393 | val_0_auc: 0.77131 |  0:00:05s
epoch 9  | loss: 0.36328 | val_0_auc: 0.78049 |  0:00:05s
epoch 10 | loss: 0.35144 | val_0_auc: 0.78763 |  0:00:06s
epoch 11 | loss: 0.32298 | val_0_auc: 0.79477 |  0:00:06s
epoch 12 | loss: 0.31852 | val_0_auc: 0.79884 |  0:00:07s
epoch 13 | loss: 0.32795 | val_0_auc: 0.79257 |  0:00:08s
epoch 14 | loss: 0.32286 | val_0_auc: 0.80158 |  0:00:08s
epoch 15 | loss: 0.32685 | val_0_auc: 0.80751 |  0:00:09s
epoch 16 | loss: 0.32004 | val_0_auc: 0.80938 |  0:00:09s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 1.05944 | val_0_auc: 0.46946 |  0:00:00s
epoch 1  | loss: 0.73135 | val_0_auc: 0.44177 |  0:00:01s
epoch 2  | loss: 0.58987 | val_0_auc: 0.47231 |  0:00:01s
epoch 3  | loss: 0.4942  | val_0_auc: 0.51802 |  0:00:02s
epoch 4  | loss: 0.47412 | val_0_auc: 0.54548 |  0:00:02s
epoch 5  | loss: 0.43199 | val_0_auc: 0.57548 |  0:00:03s
epoch 6  | loss: 0.40901 | val_0_auc: 0.58668 |  0:00:04s
epoch 7  | loss: 0.39866 | val_0_auc: 0.64359 |  0:00:04s
epoch 8  | loss: 0.39068 | val_0_auc: 0.69172 |  0:00:05s
epoch 9  | loss: 0.38024 | val_0_auc: 0.72314 |  0:00:05s
epoch 10 | loss: 0.38659 | val_0_auc: 0.73665 |  0:00:06s
epoch 11 | loss: 0.35625 | val_0_auc: 0.73665 |  0:00:07s
epoch 12 | loss: 0.36189 | val_0_auc: 0.73478 |  0:00:07s
epoch 13 | loss: 0.35469 | val_0_auc: 0.7455  |  0:00:08s
epoch 14 | loss: 0.34389 | val_0_auc: 0.73396 |  0:00:08s
epoch 15 | loss: 0.36393 | val_0_auc: 0.73709 |  0:00:09s
epoch 16 | loss: 0.34382 | val_0_auc: 0.73698 |  0:00:09s
epoch 17 | los

C:\Users\enemy\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
